# 模块 C 综合项目：CPU/GPU 通用的微型 CNN 图像分类

自己生成两类 20×20 图片，不依赖下载真实数据或预训练权重；完整实现 Dataset、训练、验证、模型保存和曲线。

**项目类型：**教师/自学参考答案 · 分步骤完整代码

## 提交要求

1. 使用 Tensor 制作两类合成图像并核对 NCHW
2. DataLoader 创建批次，构造 Conv2d + Pool + Linear
3. 用 zero_grad→forward→loss→backward→step 训练
4. 在 eval+no_grad 验证，记录损失和准确率
5. 保存 .pt 权重、训练历史 CSV、曲线 PNG

## 0. 准备工作：执行以下代码生成本地素材

这个单元格可以直接运行。素材为合成测试数据，无需联网。

In [ ]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt

torch.manual_seed(42)
out=Path("C_results");out.mkdir(exist_ok=True)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
N=160
images=torch.rand((N,1,20,20))*0.08
labels=torch.arange(N)%2
for i in range(N):
    if labels[i]==0:
        images[i,0,3:10,3:10]+=0.85
    else:
        images[i,0,10:17,10:17]+=0.85
images=images.clamp(0,1)
train_ds=TensorDataset(images[:128],labels[:128])
test_ds=TensorDataset(images[128:],labels[128:])
train_loader=DataLoader(train_ds,batch_size=16,shuffle=True)
test_loader=DataLoader(test_ds,batch_size=16)
print("设备:",device,"图片形状:",images.shape)

## TODO 1：DataLoader 创建批次，构造 Conv2d + Pool + Linear

运行参考实现，观察变量变化与输出。 

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,8,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Flatten(),nn.Linear(8*10*10,2)
        )
    def forward(self,x):
        return self.net(x)

model=SmallCNN().to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.005)
history=[]

## TODO 2：用 zero_grad→forward→loss→backward→step 训练

运行参考实现，观察变量变化与输出。 

In [ ]:
for epoch in range(1,5):
    model.train();running=0.0
    for x,y in train_loader:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad()
        loss=criterion(model(x),y)
        loss.backward();optimizer.step()
        running+=loss.item()*len(y)
    model.eval();correct=total=0
    with torch.no_grad():
        for x,y in test_loader:
            pred=model(x.to(device)).argmax(dim=1).cpu()
            correct+=int((pred==y).sum())
            total+=len(y)
    history.append({"epoch":epoch,"train_loss":running/len(train_ds),"val_accuracy":correct/total})
    print("Epoch",epoch,"loss",round(history[-1]["train_loss"],4),"acc",round(history[-1]["val_accuracy"],3))

## TODO 3：在 eval+no_grad 验证，记录损失和准确率

运行参考实现，观察变量变化与输出。 

In [ ]:
torch.save(model.cpu().state_dict(),out/"C_cnn_weights.pt")
pd.DataFrame(history).to_csv(out/"C_history.csv",index=False)
fig,ax=plt.subplots();ax.plot([h["epoch"] for h in history],[h["train_loss"] for h in history],marker="o")
ax.set_title("Training loss");ax.set_xlabel("Epoch");ax.set_ylabel("Loss")
fig.tight_layout();fig.savefig(out/"C_loss.png",dpi=150);plt.close(fig)

## TODO 4：保存 .pt 权重、训练历史 CSV、曲线 PNG

运行参考实现，观察变量变化与输出。 

In [ ]:
restored=SmallCNN()
restored.load_state_dict(torch.load(out/"C_cnn_weights.pt",map_location="cpu",weights_only=True))
with torch.no_grad():
    assert restored(images[:2]).shape==(2,2)
for name in ["C_cnn_weights.pt","C_history.csv","C_loss.png"]:
    assert (out/name).is_file() and (out/name).stat().st_size>0,name
print("C 模块全部交付文件核验通过")

## 交付核验

- `C_results/C_cnn_weights.pt`
- `C_results/C_history.csv`
- `C_results/C_loss.png`

完成后核对所有文件是否存在并可打开。参考答案在 `solutions/C_project_solution.ipynb`。

In [ ]:
# 交付文件自检：仅检查文件是否存在且非空，不自动判断模型效果
from pathlib import Path
folder=Path("C_results")
required=["C_cnn_weights.pt","C_history.csv","C_loss.png"]
missing=[name for name in required if not (folder/name).is_file() or (folder/name).stat().st_size==0]
if missing:
    print("缺失或为空的文件：",missing)
else:
    print("全部交付文件已生成")
